# Supplementary Figure 10.2: Variations in cell density and brightness.

Distribution of the number and brightness of nuclei in 1600 stitched images from 100 experiments (16 per coverslip).
The cellpose mask file was used to count the number of nuclei and to measure the mean intensity of the masked regions in the stitched images.
These results were saved to /data/nucleus-ae, which is plotted by the notebook.
The naming convention is experiment number (Run#) followed by the quarter of the coverslip (BL: bottom-left, BR: bottom-right, TL: top-left, TR: top-right) which was cropped into quarters (sixteenths) that were spelled out.

In [ ]:
#| label: sfig10b_data

%matplotlib widget

import csv
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────────────────────────────
BASE_DIR = Path("..") / "data" / "nucleus-ae"
CSV_PATH = BASE_DIR / "results.csv"

# Precomputed centroid
CENTROID_ROIS = 694.49
CENTROID_MEAN = 79.0141

# Hover sensitivity (pixels)
HOVER_THRESHOLD_PX = 8.0

# CSV column names
COL_IMAGE = "image"
COL_NUM_ROIS = "num_rois"
COL_MEAN_INTENSITY = "mean_intensity"
COL_STD_INTENSITY = "std_intensity"

# ────────────────────────────────────────────────────────────────────────

def _to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def load_csv_data(csv_path):
    """
    Load scatter plot data from CSV.
    Returns (images, num_rois, mean_intensity, std_intensity) as arrays.
    Filters out rows with NaN values.
    """
    images, num_rois, mean_vals, std_vals = [], [], [], []
    
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            img = row.get(COL_IMAGE, "")
            x = _to_float(row.get(COL_NUM_ROIS, ""))
            y = _to_float(row.get(COL_MEAN_INTENSITY, ""))
            c = _to_float(row.get(COL_STD_INTENSITY, ""))
            
            if img and np.isfinite(x) and np.isfinite(y) and np.isfinite(c):
                images.append(img)
                num_rois.append(x)
                mean_vals.append(y)
                std_vals.append(c)
    
    return (images, 
            np.asarray(num_rois), 
            np.asarray(mean_vals), 
            np.asarray(std_vals))

def create_interactive_plot(csv_path, centroid_x, centroid_y, hover_px=HOVER_THRESHOLD_PX):
    """Create interactive scatter plot with hover tooltips."""
    images, x, y, c = load_csv_data(csv_path)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(x, y, c=c, cmap="viridis", edgecolors="k")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Standard deviation intensity")
    
    # Plot centroid
    ax.scatter([centroid_x], [centroid_y], marker="x", s=200,
               c="black", linewidths=3, label="Centroid")
    ax.annotate("Centroid",
                xy=(centroid_x, centroid_y),
                xytext=(10, -10), textcoords="offset points",
                color="black", fontsize=10, fontweight="bold")
    
    ax.set_xlabel("Number of ROIs")
    ax.set_ylabel("Mean intensity")
    ax.legend(loc="upper right")
    fig.tight_layout()
    
    # Tooltip annotation (initially hidden)
    annot = ax.annotate(
        "", xy=(0, 0), xytext=(10, 10), textcoords="offset points",
        bbox=dict(boxstyle="round", fc="w", ec="0.5", alpha=0.9),
        arrowprops=dict(arrowstyle="->")
    )
    annot.set_visible(False)
    
    # KD-tree for fast hover lookup (will be rebuilt on zoom/pan)
    tree = None
    disp_xy = None
    
    def build_tree():
        nonlocal tree, disp_xy
        data_xy = np.column_stack((x, y))
        disp_xy = ax.transData.transform(data_xy)
        tree = cKDTree(disp_xy)
    
    def on_draw(event):
        build_tree()
    
    def on_move(event):
        if tree is None or not event.inaxes:
            if annot.get_visible():
                annot.set_visible(False)
                fig.canvas.draw_idle()
            return
        
        if event.x is None or event.y is None:
            return
        
        # Find nearest point in pixel space
        dist, idx = tree.query([event.x, event.y], k=1)
        if np.isfinite(dist) and dist <= hover_px:
            annot.xy = (x[idx], y[idx])
            annot.set_text(images[idx])
            if not annot.get_visible():
                annot.set_visible(True)
            fig.canvas.draw_idle()
        else:
            if annot.get_visible():
                annot.set_visible(False)
                fig.canvas.draw_idle()
    
    # Initialize tree and connect events
    build_tree()
    fig.canvas.mpl_connect("draw_event", on_draw)
    fig.canvas.mpl_connect("motion_notify_event", on_move)
    
    return fig, ax

# ────────────────────────────────────────────────────────────────────────
# Create and display plot
# ────────────────────────────────────────────────────────────────────────
fig, ax = create_interactive_plot(CSV_PATH, CENTROID_ROIS, CENTROID_MEAN)
plt.show()